# boolean-mask-identity-replace — worked example 3: Substitute a scaled identity for flagged matrices

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `boolean-mask-identity-replace`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For a batched `(B, N, N)` tensor, a `(B,)` bool mask selects whole `(N, N)` submatrices. Assigning a single `(N, N)` matrix to the masked region broadcasts that matrix across every flagged slot. `torch.eye(N) * c` builds the scaled identity used to replace degenerate slices before a batched solve or inverse.

## Worked solution

We replace each flagged `(N, N)` submatrix of a batched tensor `A` with `c * I` (a well-conditioned stand-in), non-destructively.

1. **Clone.** `out = A.clone()` protects the caller's `A`.
2. **Read `N`.** `N = A.shape[-1]` is the matrix dimension; the trailing two axes are the matrix.
3. **Build the replacement matrix.** `t.eye(N, dtype=A.dtype) * c` is an `(N, N)` matrix with `c` on the diagonal. Matching `dtype=A.dtype` avoids a dtype-promotion surprise on assignment.
4. **Masked write.** `out[mask] = repl` selects the `(K, N, N)` slice for the `K` flagged slots and assigns the single `(N, N)` matrix; it **broadcasts** across the leading `K` axis so every flagged slot receives the same scaled identity.
5. **Return `out`.** Unflagged slots are untouched.

In [ ]:
def replace_with_scaled_eye(A: Tensor, mask: Tensor, c: float) -> Tensor:
    out = A.clone()
    N = A.shape[-1]
    out[mask] = t.eye(N, dtype=A.dtype) * c
    return out

t.manual_seed(0)
A = t.randn(4, 3, 3)
mask = t.tensor([True, False, False, True])
out = replace_with_scaled_eye(A, mask, 2.0)
expected = t.eye(3) * 2.0
print('flagged -> 2*I :', bool(t.allclose(out[0], expected) and t.allclose(out[3], expected)))
print('unflagged kept :', bool(t.equal(out[1], A[1]) and t.equal(out[2], A[2])))
print('shape          :', tuple(out.shape))